# Step 1 NIRCam WFSS Reduction + Extraction w/ Grizli
## This notebook builds on grizlis WFSS tools to batch reduce NIRCam WFSS data and produce object catalogs for extraction.

For useage: Reduced files should be placed in /Prep. All other directories will be created.

This notebook is capable of reducing multiple programs, filters and GRISMs simultaneously. It can be used in conjuction with the other notebooks provided to extract spectra for a provided catalog of objects.

In [ ]:
# Import necessary modules
import os, glob, shutil, warnings, time, gc, re, traceback, subprocess, tempfile, hashlib
from pathlib import Path
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
import astropy.wcs as pywcs
import jwst
import grizli
from grizli import utils, prep, multifit, jwst_utils, grismconf

# Observation information to be reduced 
FIELD_ROOT  = 'A2744' # Prefix for all output products
MODULES = ['A', 'B'] # Which modules to use
GRISMS = ['GRISMR', 'GRISMC'] # Which GRISMs to use
FILTERS = None # Which filters to use - None = All filters
VISITS = None  # Which visits to use - Nonoe - All visits


# Group exposures
GROUP_BY = 'visit' 
COMBINE_GRISMS = False  # False: Separate reductions for GRISMR and GRISMC (Can be combined later)

PIXSCALE      = 0.04          


# Set up some directories
WORKDIR     = Path('./jwst_wfss').resolve()
RAW_DIR     = WORKDIR / 'Raw'
PREP_DIR    = WORKDIR / 'Prep'
for d in (RAW_DIR, PREP_DIR):
    d.mkdir(parents=True, exist_ok=True)


MANIFEST = WORKDIR / 'manifest.json'
dets = [f'nrc{m.lower()}long' for m in MODULES]   # Get the detector name for making groups

# Grizli/JWST configuation
os.environ['NIRCAM_CONF_VERSION'] = 'V9'
os.environ.setdefault('CRDS_PATH', str(Path.home() / 'crds_cache'))
os.environ.setdefault('CRDS_SERVER_URL', 'https://jwst-crds.stsci.edu')

print('workdir:', WORKDIR)

## 1. Group exposures by relevant keywords

In [ ]:
# Make files grizli compatible
def okey(h, k):
    """Original value of a keyword (grizli keeps 'O'+first 7 chars when it rewrites a header)."""
    return h.get('O' + k[:7], h.get(k))

# Get WCS
def pointing(f):
    with fits.open(f) as hdul:
        for ext in ('SCI', 0):
            h = hdul[ext].header
            for kr, kd in (('RA_V1', 'DEC_V1'), ('RA_REF', 'DEC_REF'), ('CRVAL1', 'CRVAL2'), ('TARG_RA', 'TARG_DEC')):
                return float(h[kr]), float(h[kd])
    return np.nan, np.nan


# Select all objects and split between GRISM images and direct images
rows = []
for f in sorted(glob.glob(str(PREP_DIR / 'jw*_rate.fits'))):
    fn = os.path.basename(f)
    if not any(d in fn for d in dets):
        continue
    if VISITS and fn[:13] not in VISITS:
        continue
    h = fits.getheader(f, 0)
    exp_type, pupil, filt = okey(h, 'EXP_TYPE'), str(okey(h, 'PUPIL')), str(okey(h, 'FILTER'))
    if pupil.startswith('GRISM') and exp_type == 'NRC_WFSS':
        kind = 'grism'
    elif exp_type == 'NRC_IMAGE':
        kind = 'direct'
    else:
        kind = 'other'
    ra, dec = pointing(f)
    rows.append(dict(file=fn, kind=kind, filter=filt, pupil=pupil, module=str(h.get('MODULE')),
                     detector=str(okey(h, 'DETECTOR')), exp_type=str(exp_type),
                     exptime=float(h.get('EFFEXPTM', np.nan)), ra=ra, dec=dec))

info = Table(rows)

is_grism = (info['kind'] == 'grism') & np.isin(info['pupil'], GRISMS)
if FILTERS:
    is_grism &= np.isin(info['filter'], FILTERS)
info = info[is_grism | (info['kind'] == 'direct')]

print(len(info), 'exposures:', {k: int((info['kind'] == k).sum()) for k in ('grism', 'direct')})
t = info.group_by(['kind', 'module', 'filter', 'pupil'])
Table(rows=[(k['kind'], k['module'], k['filter'], k['pupil'], len(g)) for k, g in zip(t.groups.keys, t.groups)],
      names=['kind', 'module', 'filter', 'pupil', 'N']).pprint_all()

## 2. Build groups for reduction

In [ ]:
KEY_LEN = {'visit': 13, 'observation': 10, 'program': 7}[GROUP_BY]
DIRECT_FILTER_MAP = {}

def label_of(fn):
    return fn[:KEY_LEN]


def cluster_by_sky(ra, dec, sep_arcsec):
    """Single-linkage clusters of pointings closer than sep_arcsec."""
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components
    n = len(ra)
    if n == 1:
        return np.zeros(1, dtype=int)
    c = SkyCoord(np.asarray(ra) * u.deg, np.asarray(dec) * u.deg)
    i1, i2, _, _ = c.search_around_sky(c, sep_arcsec * u.arcsec)
    graph = coo_matrix((np.ones(len(i1)), (i1, i2)), shape=(n, n))
    return connected_components(graph, directed=False)[1]


    return np.hypot((np.asarray(ra) - ra0) * np.cos(np.deg2rad(dec0)), np.asarray(dec) - dec0) * 3600.


def filter_number(f):
    m = re.findall(r'\d+', str(f))
    return int(m[0]) if m else 0


def select_direct(direct, label, module, wanted, ra0, dec0):
    """Return (files, filter_used) for the direct images matching a grism group."""
    pool = direct[direct['module'] == module]
    if len(pool) == 0:
        return [], None
    to_try = [DIRECT_FILTER_MAP.get(wanted, wanted)]
    others = sorted(set(pool['filter']) - set(to_try), key=lambda f: abs(filter_number(f) - filter_number(wanted)))
    to_try += others
    near = sep_arcsec(pool['ra'], pool['dec'], ra0, dec0) < 150
    same = np.array([label_of(fn) == label for fn in pool['file']])
    for f in to_try:
        m = (pool['filter'] == f)
        for extra in ([same & near]):
            sel = m & extra
            if sel.any():
                return list(pool['file'][sel]), f
    return [], None


def build_groups(info):
    grism = info[info['kind'] == 'grism']
    direct = info[info['kind'] == 'direct']

    keys = OrderedDict()
    for i in range(len(grism)):
        r = grism[i]
        pup = '' if COMBINE_GRISMS else r['pupil']
        keys.setdefault((label_of(r['file']), r['module'], r['filter'], pup), []).append(i)

    groups = []
    for (label, module, filt, pup), idx in keys.items():
        sub = grism[idx]
        cl = cluster_by_sky(sub['ra'], sub['dec'], 150)
        ncl = int(cl.max()) + 1
        for c in range(ncl):
            s = sub[cl == c]
            ra0, dec0 = float(np.mean(s['ra'])), float(np.mean(s['dec']))
            dfiles, dfilt = select_direct(direct, label, module, filt, ra0, dec0)
            g_files = list(s['file'])
            pupils = sorted(set(s['pupil']))
            root_tag = f'{label}-mod{module}-{(dfilt or filt).lower()}'
            if ncl > 1:
                root_tag += '-' + hashlib.md5('|'.join(sorted(dfiles) or sorted(g_files)).encode()).hexdigest()[:6]
            root = f'{FIELD_ROOT}-{root_tag}'
            gid = f'{root}-' + '+'.join(p.lower() for p in pupils)

            g = dict(id=gid, root=root, label=label, module=module, filter=filt, pupils=pupils,
                     grism_files=g_files, direct_files=dfiles, direct_filter=dfilt, ra=ra0, dec=dec0,
                     skip_reason=None)
            if len(dfiles) == 0:
                g['skip_reason'] = f'no {module}-module direct image in {DIRECT_FILTER_MAP.get(filt, filt)} found'
            groups.append(g)
    return groups


groups = build_groups(info)

Table(rows=[(g['id'], len(g['grism_files']), len(g['direct_files']), g['direct_filter'], g['skip_reason'] or '')
            for g in groups],
      names=['group', 'N_grism', 'N_direct', 'direct_filter', 'skip_reason']).pprint_all(max_width=200)

## 3. Initialise files with grizli

In [ ]:
def download_skyflat_https(skyfile, bucket_prefix='grizli-v2/NircamSkyflats/', **kwargs):
    try:
        f, status = utils.general_fetch_file('https://s3.amazonaws.com/' + bucket_prefix + os.path.basename(skyfile),
                                             path=os.path.dirname(skyfile), cache=True, verbose=False)
        return f if status > 0 else None
    except Exception as e:
        print('   sky flat download failed, continuing without it:', repr(e)[:100])
        return None

jwst_utils.download_skyflat_file = download_skyflat_https

_marker = PREP_DIR / '.initialized'
_done = set(_marker.read_text().split()) if _marker.exists() else set()
_raw = {p.name: p for p in RAW_DIR.rglob('*_rate.fits')}
_restored = 0
for p in sorted(PREP_DIR.glob('*_rate.fits')):
    if p.name not in _done and p.name in _raw:
        shutil.copy(_raw[p.name], p)
        _restored += 1

In [ ]:
import traceback

RESTORE = ['TELESCOP', 'INSTRUME', 'DETECTOR', 'FILTER', 'PUPIL', 'EXP_TYPE']

run_groups = [g for g in groups if not g['skip_reason']]
grism_all  = sorted({f for g in run_groups for f in g['grism_files']})
direct_all = sorted({f for g in run_groups for f in g['direct_files']})

done_marker = PREP_DIR / '.initialized'
done = set(done_marker.read_text().split()) if done_marker.exists() else set()
failed = {}
raw_copy = {p.name: p for p in RAW_DIR.rglob('*_rate.fits')}


def init_one(f):
    """Initialise one exposure. Returns None on success, otherwise a short reason (with the place in the code where it failed)."""
    try:
        ok = jwst_utils.initialize_jwst_image(f, verbose=False, oneoverf_correction=True, use_skyflats=True, nircam_edge=8)
    except Exception as e:
        fr = traceback.extract_tb(e.__traceback__)[-1]
        return f'{type(e).__name__}: {str(e)[:150]}  [{os.path.basename(fr.filename)}:{fr.lineno} in {fr.name}]'
    if not ok:
        return 'initialize_jwst_image returned a false value'
    done.add(f)
    done_marker.write_text('\n'.join(sorted(done)))
    return None


t0 = time.time()
todo = [f for f in grism_all + direct_all if f not in done]
print(len(todo), 'exposures to initialise (', len(grism_all) + len(direct_all) - len(todo), 'already done )')

os.chdir(PREP_DIR)
for i, f in enumerate(todo, 1):
    print(f'[{i}/{len(todo)}] {f}  ({time.time() - t0:.0f} s)')
    reason = init_one(f)
    if reason:
        print('   FAILED:', reason)
        failed[f] = reason


# Restore keywords in case they need to be reduced again
for f in grism_all:
    if f in failed or f not in done:
        continue
    with fits.open(f, mode='update') as hdul:
        h = hdul[0].header
        for k in RESTORE:
            ok = 'O' + k[:7]
            if ok in h:
                h[k] = h[ok]
        hdul.flush()


## 4. Build mosaics and catalogs for modelling and extraction

In [ ]:
import sep
from scipy import ndimage as nd

BAD_DQ = 1 | 2 | 1024 | 2048 | 8192 | 16384   # DO_NOT_USE, SATURATED, DEAD, HOT, LOW_QE, RC


def get_key(files, key, default=None):
    """Find a header keyword in the primary or SCI header."""
    for f in files:
        with fits.open(f) as hdul:
            for ext in (0, 'SCI'):
                if key in hdul[ext].header:
                    return hdul[ext].header[key]
    return default


def build_mosaic(root, direct_files):
    sci_file, wht_file = f'{root}_drz_sci.fits', f'{root}_drz_wht.fits'
    if os.path.exists(sci_file) and os.path.exists(wht_file):
        return sci_file, wht_file

    sci_list, wht_list, wcs_list = [], [], []
    for f in direct_files:
        with fits.open(f) as hdul:
            sci = np.ascontiguousarray(hdul['SCI'].data, dtype=np.float32)
            err = np.ascontiguousarray(hdul['ERR'].data, dtype=np.float32)
            dq = hdul['DQ'].data
            w = pywcs.WCS(hdul['SCI'].header, fobj=hdul, relax=True)
        w.pixel_shape = sci.shape[::-1]
        w.pscale = utils.get_wcs_pscale(w)

        bad = ((dq & BAD_DQ) > 0) | ~np.isfinite(sci) | ~np.isfinite(err) | (err <= 0)
        sci[bad] = 0
        bkg = sep.Background(sci, mask=bad, bw=64, bh=64, fw=3, fh=3)
        sci = (sci - bkg.back()).astype(np.float32)
        wht = np.where(bad, 0, 1.0 / np.maximum(err, 1e-12) ** 2).astype(np.float32)
        sci[bad] = 0
        sci_list.append(sci); wht_list.append(wht); wcs_list.append(w)

    fp = np.vstack([w.calc_footprint() for w in wcs_list])
    ra = ((fp[:, 0] - fp[0, 0] + 180) % 360) - 180 + fp[0, 0]
    dec = fp[:, 1]
    ra0, dec0 = 0.5 * (ra.min() + ra.max()), 0.5 * (dec.min() + dec.max())

    tmp = pywcs.WCS(naxis=2)
    tmp.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    tmp.wcs.crval = [ra0, dec0]
    tmp.wcs.cdelt = [-PIXSCALE / 3600., PIXSCALE / 3600.]
    tmp.wcs.crpix = [1., 1.]
    x, y = tmp.wcs_world2pix(ra, dec, 1)
    pad = 20
    nx = int(np.ceil(x.max() - x.min())) + 2 * pad
    ny = int(np.ceil(y.max() - y.min())) + 2 * pad

    wcs_hdr = fits.Header()
    wcs_hdr['NAXIS'] = 2
    wcs_hdr['NAXIS1'], wcs_hdr['NAXIS2'] = nx, ny
    wcs_hdr['CTYPE1'], wcs_hdr['CTYPE2'] = 'RA---TAN', 'DEC--TAN'
    wcs_hdr['CUNIT1'], wcs_hdr['CUNIT2'] = 'deg', 'deg'
    wcs_hdr['CRPIX1'] = 1. - x.min() + pad
    wcs_hdr['CRPIX2'] = 1. - y.min() + pad
    wcs_hdr['CRVAL1'], wcs_hdr['CRVAL2'] = ra0, dec0
    wcs_hdr['CD1_1'], wcs_hdr['CD1_2'] = -PIXSCALE / 3600., 0.
    wcs_hdr['CD2_1'], wcs_hdr['CD2_2'] = 0., PIXSCALE / 3600.
    wcs_hdr['RADESYS'] = 'ICRS'

    out_wcs = pywcs.WCS(wcs_hdr)
    out_wcs.pixel_shape = (nx, ny)
    out_wcs.pscale = utils.get_wcs_pscale(out_wcs)

    res = utils.drizzle_array_groups(sci_list, wht_list, wcs_list, outputwcs=out_wcs, scale=PIXSCALE,
                                     kernel='square', pixfrac=0.8, verbose=False)
    outsci, outwht = res[0], res[1]

    hdr = wcs_hdr.copy()
    for k in ['TELESCOP', 'INSTRUME', 'DETECTOR', 'FILTER', 'PUPIL', 'MODULE',
              'OTELESCOP', 'OINSTRUME', 'ODETECTO', 'OFILTER', 'OPUPIL', 'OEXP_TYP',
              'PHOTFLAM', 'PHOTPLAM', 'PHOTMJSR', 'PHOTUJA2']:
        v = get_key(direct_files, k)
        if v is not None:
            hdr[k] = v
    hdr['EXPTIME'] = float(sum(get_key([f], 'EFFEXPTM', 0.) for f in direct_files))
    hdr['BUNIT'] = 'DN/s'
    for k in ('PHOTFLAM', 'PHOTPLAM'):
        if k not in hdr:
            raise RuntimeError(f'{k} missing from the mosaic header (direct images not initialised?)')

    fits.writeto(sci_file, outsci.astype(np.float32), header=hdr, overwrite=True)
    fits.writeto(wht_file, outwht.astype(np.float32), header=hdr, overwrite=True)
    print(f'  mosaic {root}: {nx} x {ny} px from {len(direct_files)} direct exposure(s)')
    return sci_file, wht_file


def build_catalog(root, sci_file, wht_file):
    cat_file, seg_file = f'{root}.cat.fits', f'{root}_seg.fits'
    if not (os.path.exists(cat_file) and os.path.exists(seg_file)):
        prep.make_SEP_catalog(root=root, sci=sci_file, wht=wht_file, threshold=2.0, get_background=True)
    for f in (cat_file, seg_file):
        if not os.path.exists(f):
            raise RuntimeError(f'{f} not written by make_SEP_catalog: {glob.glob(root + "*")}')
    return cat_file, seg_file

In [ ]:
import json, datetime
from astropy.coordinates import SkyCoord
import astropy.units as u


def mosaic_info(sci_file, wht_file):
    """Pixel scale, sky centre and sky extent of the part of the mosaic that has weight."""
    w = pywcs.WCS(fits.getheader(sci_file))
    wht = fits.getdata(wht_file)
    ys, xs = np.where(wht > 0)
    if len(xs) == 0:
        raise RuntimeError('mosaic has no pixels with weight')
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    cx, cy = 0.5 * (x0 + x1), 0.5 * (y0 + y1)
    corners = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
    cra, cdec = w.all_pix2world([cx], [cy], 0)
    fra, fdec = w.all_pix2world([c[0] for c in corners], [c[1] for c in corners], 0)
    cen = SkyCoord(float(cra[0]) * u.deg, float(cdec[0]) * u.deg)
    radius = float(cen.separation(SkyCoord(fra * u.deg, fdec * u.deg)).arcsec.max())
    from astropy.wcs.utils import proj_plane_pixel_scales
    return dict(center_ra=float(cra[0]), center_dec=float(cdec[0]), radius_arcsec=radius,
                footprint=[[float(a), float(b)] for a, b in zip(fra, fdec)],
                pixscale=float(np.mean(proj_plane_pixel_scales(w)) * 3600.),
                shape=[int(wht.shape[1]), int(wht.shape[0])], covered_fraction=float((wht > 0).mean()))


def catalog_info(cat_file):
    cat = Table.read(cat_file)
    mag = np.asarray(cat['MAG_AUTO'], dtype=float) if 'MAG_AUTO' in cat.colnames else np.array([])
    mag = mag[np.isfinite(mag)]
    pct = [float(x) for x in np.percentile(mag, [1, 50, 99])] if len(mag) else []
    return dict(n_sources=int(len(cat)), mag_auto_1_50_99=pct)


records = []
t_all = time.time()
for gi, g in enumerate(groups, 1):
    print(f'[{gi}/{len(groups)}] {g["id"]}')
    rec = dict(g)
    rec.update(status='', note='', mosaic_sci='', mosaic_wht='', catalog='', segmentation='')
    if g['skip_reason']:
        rec.update(status='skipped', note=g['skip_reason'])
        print('   skipped:', g['skip_reason'])
        records.append(rec)
        continue
    try:
        os.chdir(PREP_DIR)
        sci_file, wht_file = build_mosaic(g['root'], g['direct_files'])
        cat_file, seg_file = build_catalog(g['root'], sci_file, wht_file)
        rec.update(mosaic_info(sci_file, wht_file))
        rec.update(catalog_info(cat_file))
        rec.update(status='ok', mosaic_sci=sci_file, mosaic_wht=wht_file, catalog=cat_file, segmentation=seg_file)
        print(f'   ok: {rec["n_sources"]} sources | mosaic {rec["shape"][0]}x{rec["shape"][1]} | radius {rec["radius_arcsec"]:.0f}"')
    except Exception as e:
        traceback.print_exc()
        rec.update(status='failed', note=repr(e)[:200])
    finally:
        os.chdir(PREP_DIR)
        gc.collect()
    records.append(rec)

# ---- merge with an existing manifest (same group id -> replaced) ----
by_id = OrderedDict()
if MANIFEST.exists():
    old = json.loads(MANIFEST.read_text())
    for r in old.get('groups', []):
        by_id[r['id']] = r
    print(f'\nmerging into existing manifest ({len(by_id)} groups)')
for r in records:
    by_id[r['id']] = r

manifest = dict(
    created=datetime.datetime.now().isoformat(timespec='seconds'),
    grizli_version=grizli.__version__,
    prep_dir=str(PREP_DIR),
    settings=dict(FIELD_ROOT=FIELD_ROOT, PIXSCALE=PIXSCALE, GROUP_BY=GROUP_BY, COMBINE_GRISMS=COMBINE_GRISMS,
                  NIRCAM_CONF_VERSION=NIRCAM_CONF_VERSION, USE_JWST_CRDS=USE_JWST_CRDS),
    groups=list(by_id.values()),
)


def _js(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


MANIFEST.write_text(json.dumps(manifest, indent=1, default=_js))
cols = ['id', 'status', 'module', 'filter', 'pupils', 'n_sources', 'center_ra', 'center_dec', 'radius_arcsec', 'note']
Table(rows=[[(', '.join(r[c]) if c == 'pupils' else r.get(c, '')) for c in cols] for r in manifest['groups']],
      names=cols).write(WORKDIR / 'groups.csv', format='ascii.csv', overwrite=True)

print(f'\nManifest: {MANIFEST} ({len(manifest["groups"])} groups, {sum(r["status"] == "ok" for r in manifest["groups"])} ok) '
      f'| {(time.time() - t_all) / 60:.1f} min this run')
Table(rows=[[(', '.join(r[c]) if c == 'pupils' else r.get(c, '')) for c in cols[:-1]] for r in manifest['groups']],
      names=cols[:-1]).pprint_all(max_width=200)
for r in manifest['groups']:
    if r['status'] in ('failed', 'skipped'):
        print(f'  {r["status"].upper():8s} {r["id"]}: {r["note"]}')